In [6]:
import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AffinityPropagation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import plotly as py
import plotly.graph_objs as go
import os
py.offline.init_notebook_mode(connected = True)
#print(os.listdir("../input"))
import datetime as dt
import missingno as msno
plt.rcParams['figure.dpi'] = 140

In [2]:
file_path = r'C:\Users\jayar\Downloads\project\Netflix-Content-Strat\Week-1-task\netflix_titles (1).csv'

df = pd.read_csv(file_path)
df.head(3)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [3]:
print(f"Number of rows: {len(df)}")

Number of rows: 8807


In [4]:
# Check for null values in the dataset
print("Null values per column:")
print(df.isnull().sum())

print("\nNull value percentages:")
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(null_pct)

# Remove rows where key content columns are missing (director, cast, country)
# This keeps rows with minor nulls in date_added/rating but removes incomplete content records
df = df.dropna(subset=['director', 'cast', 'country'])

print(f"\nDataset shape after removing rows with missing content: {df.shape}")
print(f"Rows removed: {8807 - len(df)} (only removing useless/incomplete records)")

Null values per column:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

Null value percentages:
show_id          0.00
type             0.00
title            0.00
director        29.91
cast             9.37
country          9.44
date_added       0.11
release_year     0.00
rating           0.05
duration         0.03
listed_in        0.00
description      0.00
dtype: float64

Dataset shape after removing rows with missing content: (5336, 12)
Rows removed: 3471 (only removing useless/incomplete records)


In [5]:
# Check for duplicates
print("Duplicate rows:")
print(df.duplicated().sum())

# Drop duplicates, keeping the first occurrence
df = df.drop_duplicates(keep='first')

print(f"\nDataset shape after removing duplicates: {df.shape}")
print(f"Total duplicates removed: {len(df) - df.shape[0]}")

Duplicate rows:
0

Dataset shape after removing duplicates: (5336, 12)
Total duplicates removed: 0


In [ ]:
# Check for inconsistent categories in the 'type' column
print("Unique values in 'type' column:")
print(df['type'].unique())
print(f"\nValue counts:\n{df['type'].value_counts()}")

# Standardize the 'type' column (fix inconsistencies like 'Tv Show' vs 'TV Show')
df['type'] = df['type'].str.replace('Tv Show', 'TV Show', case=False)

print(f"\nAfter standardization:")
print(df['type'].value_counts())

# Check for other categorical columns with potential inconsistencies
print("\n\nChecking 'rating' column:")
print(f"Unique ratings: {df['rating'].nunique()}")
print(f"Ratings:\n{df['rating'].value_counts()}")
# Check for inconsistencies in other categorical columns
print("\n\nChecking other categorical columns for inconsistencies:")
for col in ['rating', 'country', 'listed_in']:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"Unique values: {df[col].nunique()}")
        print(df[col].value_counts().head())

# Standardize whitespace issues (leading/trailing spaces)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
print("\n\nData cleaning complete!")
print(f"Final dataset shape: {df.shape}")
# Display summary of rows affected by data cleaning for inconsistencies in category
print("\nSummary of data cleaning:")
print(f"- Rows removed due to missing content: {8807 - len(df)}")


Unique values in 'type' column:
<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

Value counts:
type
Movie      5189
TV Show     147
Name: count, dtype: int64

After standardization:
type
Movie      5189
TV Show     147
Name: count, dtype: int64


Checking 'rating' column:
Unique ratings: 17
Ratings:
rating
TV-MA       1822
TV-14       1214
R            778
PG-13        470
TV-PG        431
PG           275
TV-G          84
TV-Y7         76
TV-Y          76
NR            58
G             40
TV-Y7-FV       3
UR             3
NC-17          2
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64


Checking other categorical columns for inconsistencies:

RATING:
Unique values: 17
rating
TV-MA    1822
TV-14    1214
R         778
PG-13     470
TV-PG     431
Name: count, dtype: int64

COUNTRY:
Unique values: 604
country
United States     1849
India              875
United Kingdom     183
Canada             107
Spain               91
Name: count, dtype: int64

L